# 08. Обучение NER (sklearn-crfsuite)

Лёгкий **CRF** на CPU с лингвистическими фичами токенов.

Метрики: token Accuracy, entity Precision/Recall/F1; learning curve.


In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (8, 4)


In [ ]:
import json
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split
from src.ner.labeling import WeakLabeler
from src.ner.features import sent2labels
from src.ner.model_crf import CRFNerModel
from src.ner.metrics import summarize_metrics

DATA = ROOT / 'файлы' / 'query_clicks.parquet'
ART = ROOT / 'artifacts'
MODELS = ROOT / 'models'
FIG = ROOT / 'figures'
for p in (ART, MODELS, FIG):
    p.mkdir(exist_ok=True)


In [ ]:
pf = pq.ParquetFile(DATA)
parts = []
seen = 0
for i in range(pf.num_row_groups):
    t = pf.read_row_group(i, columns=['toValidUTF8(query_text)'])
    df = t.to_pandas()
    df.columns = ['query']
    parts.append(df)
    seen += len(df)
    if seen >= 100000:
        break
queries = pd.concat(parts)['query'].dropna().astype(str).str.strip()
queries = queries[queries.str.len() >= 2].drop_duplicates().tolist()
print(len(queries))
labeler = WeakLabeler.from_files(ART / 'brands.txt', ART / 'categories.txt')
sents = labeler.label_dataset(queries, min_entities=1)
print('labeled', len(sents))
if len(sents) > 30000:
    rng = np.random.default_rng(42)
    sents = [sents[i] for i in rng.choice(len(sents), 30000, replace=False)]


In [ ]:
train, test = train_test_split(sents, test_size=0.2, random_state=42)
sizes, f1s, accs = [], [], []
for frac in [0.25, 0.5, 0.75, 1.0]:
    n = max(100, int(len(train) * frac))
    subset = train[:n]
    m = CRFNerModel(max_iterations=60)
    t0 = time.time()
    m.fit(subset)
    print(f'n={n} train_sec={time.time() - t0:.1f}')
    yt = [sent2labels(s) for s in test]
    yp = m.predict(test)
    r = summarize_metrics(yt, yp)
    sizes.append(n)
    f1s.append(r['micro']['f1'])
    accs.append(r['token_accuracy'])
final = CRFNerModel(max_iterations=80)
final.fit(train)
final.save(MODELS / 'ner_crf.pkl')
yt = [sent2labels(s) for s in test]
yp = final.predict(test)
report = summarize_metrics(yt, yp)
print(json.dumps(report, ensure_ascii=False, indent=2)[:1500])


In [ ]:
fig, ax = plt.subplots()
ax.plot(sizes, f1s, 'o-', color='#E31E24', label='entity micro-F1')
ax.plot(sizes, accs, 's--', color='#333', label='token Accuracy')
ax.set_title('Learning curve — CRF NER')
ax.legend()
ax.grid(True, alpha=0.3)
fig.savefig(FIG / '15_learning_curve.png', dpi=140, bbox_inches='tight')
plt.show()

per = report['per_label']
labs = list(per)
vals = [per[l]['f1'] for l in labs]
fig, ax = plt.subplots()
ax.bar(labs, vals, color='#E31E24')
ax.set_ylim(0, 1.05)
ax.set_title('NER F1 по типам сущностей')
fig.savefig(FIG / '14_ner_f1_by_entity.png', dpi=140, bbox_inches='tight')
plt.show()
